# Demo 3：Reaction to Distillation

任务：`reaction-to-distillation`。本 Demo 展示隐藏反应网络、挥发性差异和切割策略如何共同决定馏出物纯度、回收率和溶剂损失。

In [1]:
import importlib
import sys
from pathlib import Path

import pandas as pd

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src' / 'chemworld').exists()), None)
if ROOT is None:
    raise RuntimeError('请从 ChemWorld 仓库内启动 notebook')
for path in (ROOT, ROOT / 'src'):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
du = importlib.import_module('notebooks.task_demos.demo_utils')

TASK_ID = 'reaction-to-distillation'
SEED = 0

## 1. 公开任务合同

Agent 只读取公开操作、GC/final assay 反馈和任务指标，不读取 hidden state、反应拓扑或相对挥发度。

In [2]:
card = du.task_card(TASK_ID)
pd.DataFrame({
    'field': ['task_id', 'motivation', 'budget', 'episode_mode', 'success_metrics'],
    'value': [card['task_id'], card['scientific_motivation'], card['budget'], card['episode_mode'], card['reward_leaderboard_metric']['success_metrics']],
})

,field,value
0,task_id,reaction-to-distillation
1,motivation,Run a reaction and evaluate volatile-product r...
2,budget,72
3,episode_mode,campaign
4,success_metrics,"[score, distillate_purity, distillate_recovery..."


## 2. 构造候选干预

候选向量同时改变反应条件、预蒸发、蒸馏温度/时间、回流比和收集比例。

In [3]:
vectors = du.standard_probe_vectors(TASK_ID)
mid_recipe = du.recipe_frame(TASK_ID, vectors['mid'])
mid_recipe

,step,operation,payload
0,1,add_solvent,"{'operation': 'add_solvent', 'volume_L': 0.025..."
1,2,add_reagent,"{'operation': 'add_reagent', 'amount_mol': 0.0..."
2,3,add_catalyst,"{'operation': 'add_catalyst', 'catalyst_amount..."
3,4,heat,"{'operation': 'heat', 'target_temperature_K': ..."
4,5,quench,{'operation': 'quench'}
5,6,evaporate,"{'operation': 'evaporate', 'target_temperature..."
6,7,distill,"{'operation': 'distill', 'target_temperature_K..."
7,8,collect_fraction,"{'operation': 'collect_fraction', 'transfer_fr..."
8,9,measure,"{'operation': 'measure', 'instrument': 'gc'}"
9,10,terminate,{'operation': 'terminate'}


## 3. 执行并读取反馈

比较公开 `processed_estimate` 中的 `distillate_purity`、`distillate_recovery` 和 `solvent_loss`，不要把单一高纯度解释成完整成功。

In [4]:
comparison = du.compare_vectors(TASK_ID, vectors, seed=SEED)
columns = ['candidate', 'leaderboard_score', 'distillate_purity', 'distillate_recovery', 'solvent_loss', 'cost', 'all_actions_valid']
comparison[[column for column in columns if column in comparison]]

,candidate,leaderboard_score,distillate_purity,distillate_recovery,solvent_loss,cost,all_actions_valid
0,low,0.109149,0.031579,0.539601,0.202999,0.0,True
1,mid,0.221871,0.131457,0.744898,0.277999,0.0,True
2,high,0.367149,0.317529,0.901118,0.352999,1.0,True


### 查看 GC 与终检反馈

测量轨迹使 Agent 可以比较分离前后的预测，而无需访问隐藏组成。

In [5]:
run = du.run_vector(TASK_ID, vectors['mid'], seed=SEED)
measurement_feedback = du.measurement_trace(run)
measurement_feedback

,step,operation,reward,leaderboard_score,observed_keys,processed_estimate
0,9,measure,0.049487,NaN,"byproduct_signal, degradation_warning, virtual...","{'byproduct_signal': 1.0, 'degradation_warning..."
1,11,measure,0.172384,0.221871,"yield, selectivity, conversion, byproduct_sign...","{'yield': 0.14586663760997887, 'selectivity': ..."


## 4. 同一干预，不同隐藏规律

World B 使用 `topology_family` 教学控制增加隐藏反应通道。动作和观测合同不变，因此差异必须从公开产物与馏分反馈中推断。

In [6]:
paired_worlds = du.compare_hidden_worlds(
    TASK_ID, vectors['mid'], mechanism_mode='topology_family', seed=SEED
)
paired_worlds

,opaque_world,score,distillate_purity,distillate_recovery,solvent_loss,leaderboard_score,cost,safety_risk,all_actions_valid
0,World A,None,0.131457,0.744898,0.277999,0.221871,0.0,None,True
1,World B,None,0.123340,0.745241,0.277999,0.215701,0.0,None,True


## 5. 留给 Agent 的能力问题

关键不是记住一个回流比，而是判断性能变化来自反应产物分布还是蒸馏切割。

In [7]:
capability_probe = {
    'current_evidence': comparison.to_dict(orient='records'),
    'prediction_query': '预测一个未执行回流比和收集比例下的 purity/recovery/solvent-loss。',
    'next_intervention_query': '选择能区分隐藏副反应与分离参数错误的下一次实验。',
    'evaluation_note': '用未见干预预测检验 world model，而非只检查流程是否完成。',
}
capability_probe

{'current_evidence': [{'candidate': 'low',
   'score': None,
   'distillate_purity': 0.031579247284654444,
   'distillate_recovery': 0.5396013192585792,
   'solvent_loss': 0.20299850602927294,
   'leaderboard_score': 0.10914938371074265,
   'cost': 0.0,
   'safety_risk': None,
   'all_actions_valid': True,
   'operation_count': 11},
  {'candidate': 'mid',
   'score': None,
   'distillate_purity': 0.13145707278600943,
   'distillate_recovery': 0.7448976700345703,
   'solvent_loss': 0.27799850602927295,
   'leaderboard_score': 0.2218714873131021,
   'cost': 0.0,
   'safety_risk': None,
   'all_actions_valid': True,
   'operation_count': 11},
  {'candidate': 'high',
   'score': None,
   'distillate_purity': 0.3175291702184451,
   'distillate_recovery': 0.9011177929612824,
   'solvent_loss': 0.35299850602927296,
   'leaderboard_score': 0.3671485668025203,
   'cost': 1.0,
   'safety_risk': None,
   'all_actions_valid': True,
   'operation_count': 11}],
 'prediction_query': '预测一个未执行回流比和收集比例下